<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Mobilno%C5%9B%C4%87_heatmap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natężenie ruchu (Heatmap) - Historia Traffic

Ten notatnik zajmuje się tworzeniem map opóźnień ("heatmap" oraz interaktywnych punktów) w oparciu o informacje z pliku `traffic_history.csv`. Analizujemy w nim zagęszczenie i średnie opóźnienia uliczne z podziałem na godziny.

In [ ]:
# Instalacja niezbędnych bibliotek, upewnij się, że używasz odpowiedniego środowiska python.
!pip install pandas plotly osmnx matplotlib

In [ ]:
import pandas as pd
import plotly.express as px
import osmnx as ox
import matplotlib.pyplot as plt

# 1. Wczytanie i przygotowanie danych samochodowych o ruchu
df = pd.read_csv("https://raw.githubusercontent.com/aszczi/Urban_mobility_in_Cracow/refs/heads/main/traffic_history.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.strftime('%Y-%m-%d %H:00')

# Pogrupowanie danych z perspektywy krzyżówek/ulic i godzin
df_grouped = df.groupby(['hour', 'point_name', 'lat', 'lon'], as_index=False).agg({
    'delay_sec': 'mean',
    'congestion_index_pct': 'mean',
    'current_speed_kmph': 'mean'
})

df_grouped = df_grouped.sort_values(by=['hour'])
df_grouped.head()

## Interaktywna mapa względem godziny

Wykorzystujemy `scatter_mapbox` jako formę interaktywnej "heatmapy". Animacja pokazuje zmiany w natężeniu ruchu ulicznego z biegiem czasu.

In [ ]:
# Interaktywna heatmapa dla godzin (pozostawiona jako podgląd punktowy, gdyż rysowanie całych linii za każdym razem 
# w animacji plotly bywa sprzętowo wymagające)

fig = px.scatter_mapbox(
    df_grouped,
    lat="lat",
    lon="lon",
    color="congestion_index_pct", 
    size="delay_sec",             
    hover_name="point_name",      
    hover_data={"delay_sec": True, "congestion_index_pct": True, "current_speed_kmph": True, "lat": False, "lon": False},
    animation_frame="hour",       
    color_continuous_scale=px.colors.sequential.Inferno, 
    range_color=[0, 100],         
    zoom=11,                      
    center={"lat": 50.0647, "lon": 19.9450}, 
    title="Zmienne natężenie / opóźnienia w ruchu kołowym w Krakowie (względem godziny)",
    height=800
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

## Statyczna mapa z siatką drogową OSMnx

Naniesienie punktów bez tła dynamicznego bezprośrednio na układ dróg z biblioteki statycznej.

In [ ]:
import matplotlib.colors as mcolors

# 1. Pobieranie grafu drogowego dla Krakowa (drive - tylko dla pojazdów)
lokalizacja = "Kraków, Poland"
print(f"Pobieranie geometrii dróg dla: {lokalizacja}... To może zająć chwilkę.")
G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)

# 2. Wybieramy dane z określonej godziny (np. pierwsza godzina z DataFrame)
first_hour_str = df_grouped['hour'].iloc[0]
df_hour = df_grouped[df_grouped['hour'] == first_hour_str]

# 3. Przypisywanie przestrzenne (Spatial mapping)
# Wyszukujemy krawędzie dróg znajdujące się najbliżej punktów natężenia.
lons = df_hour['lon'].values
lats = df_hour['lat'].values

print("Dopasowywanie punktów do okolicznych ulic (nearest_edges)...")
# W starszych wersjach osmnx bywa ox.get_nearest_edges, w nowych ox.nearest_edges lub ox.distance.nearest_edges
nearest_edges = ox.distance.nearest_edges(G, X=lons, Y=lats)

# Tworzymy słownik krawędź -> congestion_index_pct
# (u, v, key) to identyfikator osi ulicy (krawędzi) w bibliotece OSMnx
edge_congestion = {}
for i, edge in enumerate(nearest_edges):
    congestion = df_hour['congestion_index_pct'].iloc[i]
    edge_congestion[edge] = congestion

# 4. Ustalamy kolory i grubość dla każdej ulicy na mapie
edge_colors = []
edge_linewidths = []

cmap = plt.get_cmap('autumn_r') # Czerwono-pomarańczowo-żółty (czerwień = najgorzej)
norm = mcolors.Normalize(vmin=0, vmax=100) # Konfigurujemy natężenie 0-100%

for u, v, k, data in G.edges(keys=True, data=True):
    edge_tuple = (u, v, k)
    if edge_tuple in edge_congestion:
        # Pomalujmy pokryte ulice na kolor zgodny z opóźnieniami
        val = edge_congestion[edge_tuple]
        color = mcolors.to_hex(cmap(norm(val)))
        edge_colors.append(color)
        edge_linewidths.append(4.0) # Pogrubienie objętych ulic
    else:
        # Standardowe pozostałe ulice, szare
        edge_colors.append('#333333')
        edge_linewidths.append(0.5)

# 5. Wykreślenie siatki dróg z efektem zbliżonym do linii w nawigacjach samochodowych Google
print("Generowanie rysunku mapy...")
fig, ax = ox.plot_graph(
    G, 
    node_size=0, # Brak kółek punktowych dla węzłów
    edge_linewidth=edge_linewidths, 
    edge_color=edge_colors, 
    bgcolor="#111111", # Ciemne tło na którym łatwiej zobaczyć kolorowe linie
    show=False, 
    close=False,
    figsize=(14, 14)
)

# Dodanie legendy dla skali koloru
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label('Gęstość ruchu (Congestion %)', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
cbar.ax.set_yticklabels(cbar.ax.get_yticks(), color='white')

plt.title(f"Natężenie ruchu na geometriach ulic dla godziny: {first_hour_str}", color='white', fontsize=16)
plt.show()